# M3L3 E18 — Sistema SaaS multiagente
### Módulo 3 · Lecture 3 · Sistemas Multiagente Avanzados

**Caso terminado:** plataforma SaaS con agentes especializados por departamento — producto, soporte técnico y facturación.

## ¿Qué vas a ver en este ejercicio?
- Un sistema multiagente que cubre los 3 departamentos principales de una empresa SaaS.
- Cada agente usa RAG sobre su propia knowledge base departamental.
- Router LLM que identifica a qué departamento pertenece la consulta del cliente.

## Arquitectura del sistema

> **Sistema SaaS multiagente:** cada departamento tiene su propio agente con conocimiento especializado. El router decide quién atiende al cliente antes de que llegue al especialista.

```
START
  |
  v
router_company_node
  |         |          |          |
  v         v          v          v
product  tech_support  billing  fallback
  |         |          |          |
  +----+----+----------+----------+
       |
      END
```

| Departamento | Agente | Tipo de consultas |
|---|---|---|
| Producto | `product_rag_agent` | Features, roadmap, planes, onboarding |
| Soporte técnico | `technical_support_rag_agent` | Bugs, errores, APIs, integraciones |
| Facturación | `billing_rag_agent` | Facturas, upgrades, cancelaciones |
| Fallback | `fallback_agent` | Consultas no clasificadas |

## Paso 1 — Elegí tu proveedor de LLM

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

## Paso 2 — Instalar LangGraph

In [ ]:
!pip install langgraph -q

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph listo.")

## Sección 1 — Knowledge bases departamentales

> **Knowledge base departamental:** cada agente tiene acceso exclusivo a la información de su área. Esto evita que el agente de facturación responda preguntas técnicas y viceversa.

In [ ]:
product_kb = [
    "Features principales: dashboard en tiempo real, reportes automáticos, API REST y webhooks.",
    "Próximas features en roadmap Q1: integración con IA generativa, exportación a Power BI.",
    "Onboarding: proceso guiado de 5 pasos, video tutoriales, base de conocimiento con 200+ artículos.",
    "Plan Starter incluye features básicas. Plan Pro desbloquea reportes avanzados y automatizaciones.",
    "Plan Enterprise incluye features personalizadas, white-labeling y entornos separados dev/prod.",
    "Límite de API: 1.000 requests/hora en Starter, 10.000 en Pro, ilimitado en Enterprise.",
    "Mobile app disponible para iOS y Android con sincronización en tiempo real.",
]

technical_support_kb = [
    "Error 401 Unauthorized: verificar que el API key esté activo y tenga los permisos correctos.",
    "Error 429 Rate Limit: reducir la frecuencia de llamadas o actualizar a un plan con mayor límite.",
    "Webhooks no llegan: verificar URL pública accesible desde internet, revisar logs en el dashboard.",
    "Dashboard no carga: limpiar caché del navegador, deshabilitar extensiones, probar en modo incógnito.",
    "Integración con Slack: instalar la app desde el Marketplace, autorizar con OAuth, configurar canal.",
    "SLA: tiempo de respuesta garantizado 1 hora en Enterprise, 4 horas en Pro, 24 horas en Starter.",
    "Status page disponible en status.nuestra-plataforma.com para ver incidentes en tiempo real.",
]

billing_kb = [
    "Facturación mensual o anual. Anual tiene 20% de descuento sobre el precio mensual.",
    "Métodos de pago aceptados: tarjeta de crédito/débito (Visa, Mastercard, Amex), transferencia bancaria.",
    "Upgrade de plan: efectivo inmediatamente, se cobra la diferencia prorrateada del mes en curso.",
    "Downgrade de plan: efectivo al próximo ciclo de facturación, sin cargo adicional.",
    "Cancelación: se puede cancelar en cualquier momento, el acceso continúa hasta fin del período pagado.",
    "Reembolsos: disponibles dentro de los primeros 30 días por política de satisfacción garantizada.",
    "Facturas disponibles en el portal de clientes, se envían automáticamente por email al responsable de cuenta.",
]

knowledge_bases = {
    "product": product_kb,
    "technical_support": technical_support_kb,
    "billing": billing_kb,
}

print("Knowledge bases cargadas:", list(knowledge_bases.keys()))

## Sección 2 — State del sistema

In [ ]:
class CompanyState(TypedDict):
    query: str
    department: str   # "product" | "technical_support" | "billing" | "unknown"
    reason: str
    response: str

## Sección 3 — Nodos del sistema

> **Agente RAG:** el patrón consiste en (1) recuperar el contexto relevante de la KB, (2) construir un prompt con ese contexto, (3) invocar el LLM. El resultado es una respuesta fundamentada en datos reales.

In [ ]:
def router_company_node(state: CompanyState) -> dict:
    prompt = (
        "Sos el router de soporte de una empresa SaaS.\n"
        "Clasificá la consulta del cliente en exactamente un departamento:\n"
        "- 'product': features, funcionalidades, roadmap, onboarding, cómo usar la plataforma\n"
        "- 'technical_support': errores, bugs, APIs, integraciones, problemas técnicos\n"
        "- 'billing': facturas, pagos, planes, upgrades, cancelaciones, reembolsos\n"
        "- 'unknown': consultas que no corresponden a ningún departamento\n\n"
        "Respondé con JSON: {\"department\": \"...\", \"reason\": \"...\"}\n"
        "Solo department y reason, sin markdown.\n\n"
        f"Consulta: {state['query']}"
    )
    response = llm.invoke(prompt)
    import json, re
    text = response.content.strip()
    text = re.sub(r"```[\w]*\n?", "", text).strip()
    try:
        data = json.loads(text)
        department = data.get("department", "unknown").lower()
        reason = data.get("reason", "")
    except Exception:
        department = "unknown"
        reason = text
    if department not in ("product", "technical_support", "billing"):
        department = "unknown"
    return {"department": department, "reason": reason}


def _rag_agent(department: str, role: str, state: CompanyState) -> dict:
    context = "\n".join(knowledge_bases[department])
    response = llm.invoke(
        f"Sos el agente de {role} de una empresa SaaS.\n"
        "Respondé la consulta del cliente usando únicamente el contexto provisto.\n"
        "Si la información no está en el contexto, indicalo claramente.\n\n"
        f"Contexto de {role}:\n{context}\n\n"
        f"Consulta: {state['query']}\n\n"
        "Respondé de forma clara y concisa en español."
    )
    return {"response": response.content.strip()}


def product_rag_agent(state: CompanyState) -> dict:
    return _rag_agent("product", "Producto", state)


def technical_support_rag_agent(state: CompanyState) -> dict:
    return _rag_agent("technical_support", "Soporte Técnico", state)


def billing_rag_agent(state: CompanyState) -> dict:
    return _rag_agent("billing", "Facturación", state)


def fallback_agent(state: CompanyState) -> dict:
    return {
        "response": (
            "Gracias por contactarnos. Para poder ayudarte mejor, ¿podés indicarnos si tu consulta "
            "es sobre: (1) funcionalidades de la plataforma, (2) un problema técnico, o (3) facturación y pagos? "
            "También podés escribirnos a soporte@empresa.com."
        )
    }


def department_router(state: CompanyState) -> str:
    return {
        "product": "product_rag_agent",
        "technical_support": "technical_support_rag_agent",
        "billing": "billing_rag_agent",
    }.get(state["department"], "fallback_agent")


print("Nodos definidos.")

## Sección 4 — Compilar el grafo

In [ ]:
graph = StateGraph(CompanyState)

graph.add_node("router_company_node",         router_company_node)
graph.add_node("product_rag_agent",           product_rag_agent)
graph.add_node("technical_support_rag_agent", technical_support_rag_agent)
graph.add_node("billing_rag_agent",           billing_rag_agent)
graph.add_node("fallback_agent",              fallback_agent)

graph.add_edge(START, "router_company_node")
graph.add_conditional_edges(
    "router_company_node",
    department_router,
    {
        "product_rag_agent":           "product_rag_agent",
        "technical_support_rag_agent": "technical_support_rag_agent",
        "billing_rag_agent":           "billing_rag_agent",
        "fallback_agent":              "fallback_agent",
    },
)
for node in ["product_rag_agent", "technical_support_rag_agent", "billing_rag_agent", "fallback_agent"]:
    graph.add_edge(node, END)

app = graph.compile()
print("Grafo compilado.")

## Demo — Consultas de clientes SaaS

El sistema detecta el departamento correcto y genera respuestas con información real de la KB.

In [ ]:
EMPTY = {"query": "", "department": "", "reason": "", "response": ""}

queries = [
    "¿Cuándo van a lanzar la integración con IA?",
    "Me aparece error 429, ¿qué significa?",
    "Quiero cancelar mi suscripción y que me devuelvan el dinero",
    "¿Cómo configuro los webhooks para que lleguen a mi servidor?",
    "¿Tienen oficinas en Buenos Aires?",
]

for q in queries:
    r = app.invoke({**EMPTY, "query": q})
    print(f"\nConsulta:    {q}")
    print(f"Departamento: {r['department']}")
    print(f"Respuesta:    {r['response'][:120]}...")
    print("-" * 70)

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    empty = {"query": "", "department": "", "reason": "", "response": ""}

    r1 = app.invoke({**empty, "query": "¿qué features tiene el plan Enterprise?"})
    assert r1["department"] == "product", f"esperaba product: {r1['department']}"
    assert len(r1["response"]) > 10

    r2 = app.invoke({**empty, "query": "el webhook no está llegando a mi servidor"})
    assert r2["department"] == "technical_support", f"esperaba technical_support: {r2['department']}"
    assert len(r2["response"]) > 10

    r3 = app.invoke({**empty, "query": "¿cómo cambio el método de pago de mi cuenta?"})
    assert r3["department"] == "billing", f"esperaba billing: {r3['department']}"
    assert len(r3["response"]) > 10

    r4 = app.invoke({**empty, "query": "¿cuántos empleados tiene la empresa?"})
    assert r4["department"] == "unknown", f"esperaba unknown: {r4['department']}"

    print("Checks E18 OK")

run_checks()

## ¿Qué viste en este caso?

- Un sistema SaaS real tiene departamentos con conocimiento especializado — ningún agente genérico podría reemplazarlos.
- El patrón `_rag_agent(department, role, state)` evita duplicar código entre agentes.
- El router LLM es mucho más robusto que keywords: entiende que "error 429" es soporte técnico, no producto.

| Aspecto | Detalle |
|---|---|
| Router | LLM con JSON output → `{department, reason}` |
| RAG | `\"\\n\".join(kb)` inyectado en el prompt del agente |
| Reutilización | `_rag_agent()` helper evita duplicación |
| Departamentos | 3 especializados + 1 fallback |

## Próximo ejercicio

En **E19** vas a ver un sistema empresarial completo con 4 departamentos (HR, IT, Finance, Legal) más un **agente evaluador** que audita la calidad de cada respuesta.